In [1]:
import pandas as pd
import numpy as np

### Coordinate System Translator

- standardizes the three coordinate systems so that everything we do here can be translated into either system

In [ ]:
class CoordSys():

    dcs = "dcs"
    avl = "avl"
    blender = "bld"
    traditional = "trd"

    # DCS
    #       x: length axis,     nose(+) to tail(-)
    #       y: height axis,     top(+) to bottom(-)
    #       z: span axis,       left wingtip(-) to right wingtip(+)

    # BLENDER
    #       x: length axis,     nose(+) to tail(-)
    #       y: span axis,       left wingtip(+) to right wingtip(-)
    #       z: height axis,     top(+) to bottom(-)

    # AVL
    #       x: length axis,     nose(-) to tail(+)
    #       y: span axis,       left wingtip(-) to right wingtip(+)
    #       z: height axis,     top(+) to bottom(-)

    def coordsToBlender(coords, coord_sys_name):
        if coord_sys_name == CoordSys.dcs:
            return CoordSys.coordDCSToBlender(coords)
        
    def coordsBlenderToAvl(coords):
        x, y, z = 0, 1, 2
        return np.array([-coords[0], -coords[1], coords[2]])

    def coordDCSToBlender(coords):
        x, y, z = 0, 1, 2
        return np.array([coords[0], -coords[2], coords[1]])



### Aerodynamic Functions Class

- get chord length (leading edge point, trailing edge point)

In [49]:
class AeroHelper():

    # Assumes all points are blender coord sys points

    x = 0
    y = 1
    z = 2

    # input: blender coords inboard pt (X, Y, Z) outboard pt (X, Y, Z)
    # purpose: computes the absolute distance between two points
    @staticmethod
    def getDistance(inboard_pt, outboard_pt):
        return np.linalg.norm(inboard_pt - outboard_pt)
    
    # Inputs: blender coords line_pt_A, line_pt_B, C each shape (3,)
    # purpose:
    #   Finds the intersection of the perpendicular from C onto the infinite line through A-B
    #   and calculates it's length
    @staticmethod
    def getPerpendicularDistance(line_pt_A, line_pt_B, C):
        v = line_pt_B  - line_pt_A  # direction of line AB
        if np.allclose(v, 0):
            # degenerate line (A == B): foot is A itself
            P = line_pt_A.copy()
        else:
            w = C - line_pt_A
            t = np.dot(w, v) / np.dot(v, v)  # scalar projection parameter
            P = line_pt_A + t * v
        perp_vec = P - C
        distance = np.linalg.norm(perp_vec)

        print(f"coordinate P is {P}")

        return distance
    
    @staticmethod
    def getIntegratedMAC(panels):
        return panels

### PanelDTO
- contains the data for a single panel of the wing.
- contains the functions to calculate stats for a single panel of the wing

In [50]:
class PanelDto(object):

    le_inboard_pt = None
    le_outboard_pt = None
    te_inboard_pt = None
    te_outboard_pt = None

    inboard_chord_length = 0
    outboard_chord_length = 0

    span_width = 0

    # Inputs: 
    #   nparray(leading edge inboard point)
    #   nparray(trailing edge inboard point)
    #   nparray(leading edge outboard point)
    #   nparray(trailing edge outboard point)

    def __init__(self, le_inboard_pt, te_inboard_pt, le_outboard_pt, te_outboard_pt):
        self.le_inboard_pt = le_inboard_pt
        self.le_outboard_pt = le_outboard_pt
        self.te_inboard_pt = te_inboard_pt
        self.te_outboard_pt = te_outboard_pt

        self.inboard_chord_length = self.getChordLength(self.le_inboard_pt, self.te_inboard_pt)
        self.outboard_chord_length = self.getChordLength(self.le_inboard_pt, self.te_inboard_pt)

        self.span_width = self.getSpanWidth(self.le_inboard_pt, self.te_inboard_pt, 
                                            self.le_outboard_pt, self.te_outboard_pt)

    def getChordLength(self, le_pt, te_pt):
        # all points must be in blender coordinates

        # neither points can be null
        assert(le_pt.any())
        assert(te_pt.any())

        # coordinate point shape must be (3,)
        assert(le_pt.shape == (3,))
        assert(te_pt.shape == (3,))

        # coordinate point's span values should be about the same
        assert(le_pt[1] - te_pt[1] < 0.01)

        return AeroHelper.getDistance(le_pt, te_pt)
    
    def getSpanWidth(self, le_ib_pt, te_ib_pt, le_ob_pt, te_ob_pt):
        assert(le_ib_pt.any())
        assert(te_ib_pt.any())
        assert(le_ob_pt.any())
        assert(te_ob_pt.any())

        assert(le_ib_pt.shape == (3,))
        assert(te_ib_pt.shape == (3,))
        assert(le_ob_pt.shape == (3,))
        assert(te_ob_pt.shape == (3,))

        le_width = AeroHelper.getPerpendicularDistance(le_ib_pt, te_ib_pt, le_ob_pt)
        te_width = AeroHelper.getPerpendicularDistance(le_ib_pt, te_ib_pt, le_ob_pt)

        if le_width - te_width < 0.01:
            return le_width
        else:
            print(f"Logger - Warning: LE width ({le_width}) different from TE width ({te_width})")
            return (le_width + te_width) / 2
    
    

In [51]:
le_ib_pt = np.array([3.756745, 1.996377, 0.327547])
te_ib_pt = np.array([-3.52977, 1.99341, 0.318274])
le_ob_pt = np.array([2.891615, 2.068156, 0.373958])
te_ob_pt = np.array([-3.597471, 2.067886, 0.316832])

panel = PanelDto(le_ib_pt, te_ib_pt, le_ob_pt, te_ob_pt)

print(panel.inboard_chord_length)
print(panel.outboard_chord_length)
print(panel.span_width)

coordinate P is [2.89170484 1.99602476 0.32644613]
coordinate P is [2.89170484 1.99602476 0.32644613]
7.286521504589347
7.286521504589347
0.08637303540828731


### Calculate Wing MAC

In [ ]:
def getMAC(wing_panels):

    half_wing_area = 0
    weighted_chord_sum = 0
    weighted_LE_sum = 0

    for panel in wing_panels:
        # 1.
        # compute inboard chord length
        # compute outboard chord length

        inboard_chord_length = panel.inboard_chord_length
        outboard_chord_length = panel.outboard_chord_length

        # 2.
        # compute spanwise width

        panel_width = panel.span_width

        # 3.
        # approximate area's contribution

